Topic 121 | k-NN - Distance Metrics

In [2]:
import numpy as np
from sklearn.neighbors  import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

# create a simple data set
X,Y = make_classification(
    n_samples= 1000,
    n_informative=2,
    n_redundant=0,
    n_features=2,
    random_state=42
)
x_train,x_test, y_train,y_test = train_test_split(
    X,Y, test_size=0.2, random_state=42
)


In [5]:
# train k-nn with different distance matrices
from sklearn import metrics

metrics = {
    "Euclidean": KNeighborsClassifier(n_neighbors=5, metric='euclidean'),
    "Manhattan": KNeighborsClassifier(n_neighbors=5, metric='manhattan'),
    "Minkowski (p=3)": KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=3)
}

for name, model in metrics.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    acc = accuracy_score(y_test, y_pred)
    print(f'{name} accuracy:{acc:.3f}')
    

Euclidean accuracy:0.935
Manhattan accuracy:0.950
Minkowski (p=3) accuracy:0.940


In [6]:
# show numerical example of distance matrices
x= np.array([2,3])
x1 = np.array([5,7])

euclidean_distance = np.linalg.norm(x-x1)
manhattan_distance = np.sum(np.abs(x-x1))
minkowski_distance = np.power(np.sum(np.abs(x-x1)**3), 1/3)

print('distance exmapel of between xand x1')
print(f'Euclidean distance: {round(euclidean_distance, 3)}')
print(f'Manhattan distance: {round(manhattan_distance, 3)}')
print(f'Minkowski distance: {round(minkowski_distance, 3)}')


distance exmapel of between xand x1
Euclidean distance: 5.0
Manhattan distance: 7
Minkowski distance: 4.498


Topic 122 | k-NN - Performance Optimization Techniques

In [8]:
# optimizing k-NN performance  : k selection , normalization and PCA
import numpy as np 
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# load datasets
data = load_breast_cancer()
X,Y= data.data, data.target

x_train, x_test, y_train, y_test=train_test_split(
    X,Y, test_size=0.25, random_state=42 , stratify=Y
)

print("origanal demension ", x_train.shape)
# normalize the data
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)


origanal demension  (426, 30)


In [10]:
# chose optiaml k (simaple search)
best_k =None
acc_k=0
for k in range(1,21):
    knn= KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(x_train_scaled, y_train)
    y_pred = knn.predict(x_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    print(f'k: {k}, accuracy: {acc:.3f}')
    
    if acc > acc_k:
        acc_k = acc
        best_k = k

print(f'best k: {best_k} with accuracy: {acc_k:.3f}')


k: 1, accuracy: 0.951
k: 2, accuracy: 0.937
k: 3, accuracy: 0.972
k: 4, accuracy: 0.958
k: 5, accuracy: 0.979
k: 6, accuracy: 0.972
k: 7, accuracy: 0.979
k: 8, accuracy: 0.972
k: 9, accuracy: 0.965
k: 10, accuracy: 0.965
k: 11, accuracy: 0.972
k: 12, accuracy: 0.979
k: 13, accuracy: 0.958
k: 14, accuracy: 0.965
k: 15, accuracy: 0.958
k: 16, accuracy: 0.958
k: 17, accuracy: 0.951
k: 18, accuracy: 0.951
k: 19, accuracy: 0.951
k: 20, accuracy: 0.951
best k: 5 with accuracy: 0.979


In [11]:
#demension reduction with PCA
pca = PCA(n_components=10)
x_train_pca = pca.fit_transform(x_train_scaled)
x_test_pca = pca.transform(x_test_scaled)

print("reduced demension ", x_train_pca.shape)

#train kNN after PCA using the best k
knn_pca = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')
knn_pca.fit(x_train_pca, y_train)

y_pred_pca = knn_pca.predict(x_test_pca)
acc_pca = accuracy_score(y_test, y_pred_pca)

print(f'accuracy after PCA: {acc_pca:.3f}')
print("explained variance ratio of PCA components: ", round(np.sum(pca.explained_variance_ratio_), 3))

reduced demension  (426, 10)
accuracy after PCA: 0.979
explained variance ratio of PCA components:  0.954


Topic 123 | k-NN - Implementation with Scikit-Learn

In [12]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,confusion_matrix
# load the digits dataset
digits = load_digits()
X, y = digits.data, digits.target
# split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# feature scaling importance for distance-based algorithms
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

print ("training sample ", x_train.shape[0])
print ("featured per sample", x_train.shape[1])

training sample  1437
featured per sample 64


In [13]:
#train knn model
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn.fit(x_train, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'euclidean'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [14]:
# make prediction
y_pred = knn.predict(x_test)

#evaluate perforamnce
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
print(f'accuracy: {acc:.3f}')
print("confusion matrix: \n", cm)

accuracy: 0.983
confusion matrix: 
 [[36  0  0  0  0  0  0  0  0  0]
 [ 0 36  0  0  0  0  0  0  0  0]
 [ 0  0 35  0  0  0  0  0  0  0]
 [ 0  0  0 37  0  0  0  0  0  0]
 [ 0  0  0  0 36  0  0  0  0  0]
 [ 0  0  0  0  0 37  0  0  0  0]
 [ 0  0  0  0  0  0 35  0  1  0]
 [ 0  0  0  0  0  0  0 36  0  0]
 [ 0  2  0  0  0  0  0  1 32  0]
 [ 0  0  0  0  1  0  0  0  1 34]]
